# Step 04 — Regime Profiling

This notebook explores the regime profiles produced by `pipelines/04_regime_label.py`.
Each cluster is assigned an economic name (e.g. Stagflation, Growth Boom) and
characterised by its distribution of key macro indicators.

Key outputs:
- **Regime timeline** — which regime was active each quarter
- **Transition matrix** — probability of moving from one regime to another
- **Indicator profiles** — box-plot distributions of key variables per regime

**Run `python pipelines/04_regime_label.py` before executing this notebook.**

## Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import logging

import pandas as pd
import yaml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import DATA_DIR
from market_regime import plotting

setup_logging("INFO")
log = logging.getLogger("04_regimes")

cfg = load()

run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

print("RunConfig:", run_cfg)
print("DATA_DIR: ", DATA_DIR)

## Load Cluster Labels, Features, Regime Names, and Transition Matrix

In [ ]:
REGIMES_DIR = DATA_DIR / "regimes"
PROCESSED_DIR = DATA_DIR / "processed"

labels = None
features = None
tm = None
regime_names = {}

# Cluster labels
try:
    labels_df = pd.read_parquet(REGIMES_DIR / "cluster_labels.parquet")
    # Use balanced_cluster as the primary assignment
    if "balanced_cluster" in labels_df.columns:
        labels = labels_df["balanced_cluster"]
    elif "cluster" in labels_df.columns:
        labels = labels_df["cluster"]
        print("Note: using 'cluster' (balanced_cluster not found)")
    print(f"Labels loaded: {len(labels)} quarters")
except FileNotFoundError:
    print("ERROR: cluster_labels.parquet not found. Run pipelines/03_cluster.py first.")

# Feature matrix
try:
    features = pd.read_parquet(PROCESSED_DIR / "features.parquet")
    print(f"Features loaded: {features.shape}")
except FileNotFoundError:
    print("ERROR: features.parquet not found. Run pipelines/02_features.py first.")

# Transition matrix
try:
    tm = pd.read_parquet(REGIMES_DIR / "transition_matrix.parquet")
    print(f"Transition matrix loaded: {tm.shape}")
except FileNotFoundError:
    print("ERROR: transition_matrix.parquet not found. Run pipelines/04_regime_label.py first.")

## Load Regime Names

Suggested names are stored in `data/regimes/regime_names_suggested.yaml`
after running `pipelines/04_regime_label.py`. Edit `config/regime_labels.yaml`
to override with manually pinned names.

In [ ]:
# Try suggested names first, then fall back to manually pinned labels
names_candidates = [
    REGIMES_DIR / "regime_names_suggested.yaml",
    DATA_DIR.parent / "config" / "regime_labels.yaml",
]

for path in names_candidates:
    try:
        with open(path) as f:
            raw_names = yaml.safe_load(f)
        # Normalize keys to int
        regime_names = {int(k): str(v) for k, v in raw_names.items()}
        print(f"Regime names loaded from {path}:")
        for k, v in sorted(regime_names.items()):
            print(f"  Regime {k}: {v}")
        break
    except FileNotFoundError:
        continue
    except Exception as exc:
        print(f"Warning: could not load {path}: {exc}")
        continue

if not regime_names:
    print("No regime names file found — using generic names (Regime 0, Regime 1, ...)")
    if labels is not None:
        unique = sorted(labels.dropna().astype(int).unique())
        regime_names = {i: f"Regime {i}" for i in unique}

## Regime Timeline

Horizontal strip chart showing which regime was active each quarter.
Each row is one regime; coloured bands indicate active periods.

In [ ]:
if labels is not None:
    plotting.plot_regime_timeline(labels, regime_names, run_cfg)

## Transition Matrix

Heatmap of regime transition probabilities. The diagonal represents regime
persistence (probability of staying in the same regime next quarter).
High off-diagonal values indicate common regime transitions.

In [ ]:
if tm is not None:
    plotting.plot_transition_matrix(tm, regime_names, run_cfg)

## Key Indicator Profiles

Box plots for a set of key economic indicators across regimes.
These distributions help validate that the regime names assigned
in the previous step are economically meaningful.

In [ ]:
KEY_INDICATORS = ["us_infl", "10yr_ustreas", "credit_spread", "gdp_growth", "sp500_pe", "us_pop_growth"]

if features is not None and labels is not None:
    # Report which requested columns are actually present
    available = [c for c in KEY_INDICATORS if c in features.columns]
    missing   = [c for c in KEY_INDICATORS if c not in features.columns]
    if missing:
        print(f"Note: requested indicator columns not found: {missing}")
        print("Using available columns only.")
    if available:
        # Align labels to features index
        aligned_labels = labels.reindex(features.index)
        plotting.plot_regime_profiles(features, aligned_labels, regime_names, available, run_cfg)
    else:
        print("None of the requested key indicators found in features. Available columns:")
        print(list(features.columns[:20]))

## Transition Matrix Table

In [ ]:
if tm is not None:
    print("Regime Transition Probability Matrix:")
    print("(rows = current regime, columns = next regime)")
    print()
    # Rename index/columns to regime names for readability
    tm_named = tm.copy()
    tm_named.index   = [regime_names.get(int(i), f"R{i}") for i in tm.index]
    tm_named.columns = [regime_names.get(int(c), f"R{c}") for c in tm.columns]
    display(tm_named.style.format("{:.3f}").background_gradient(cmap="Blues", axis=None))

## Regime Summary

In [ ]:
if labels is not None:
    counts = labels.dropna().astype(int).value_counts().sort_index()
    total  = counts.sum()

    print(f"{'ID':>4}  {'Name':<35}  {'Count':>6}  {'Pct':>7}")
    print("-" * 58)
    for cid, count in counts.items():
        name = regime_names.get(cid, f"Regime {cid}")
        pct  = count / total * 100
        print(f"  {cid:>2}  {name:<35}  {count:>6}  {pct:>6.1f}%")
    print("-" * 58)
    print(f"  {'':>2}  {'TOTAL':<35}  {total:>6}  {100.0:>6.1f}%")